[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalo/TorchCode/blob/master/solutions/00_p04_indexing_masking_solution.ipynb)

# 📖 Solution: Indexing & Masking

Reference solutions for every exercise in `00_p04_indexing_masking.ipynb`. Try the template first!

### 📐 Notation

| Symbol | Meaning |
|:---:|---|
| `B` | batch size |
| `C` | number of classes (`pick_per_row`) |
| `V` | vocabulary size (`keep_topk`, `unsort`) |
| `S_q` | number of queries |
| `S_k` | number of keys |
| `n` | sequence length for `attention_mask` |
| `w` | window size |
| `G` | number of groups (`group_mean`) |

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

## 1. Slicing

Slices return **views** (no copy). A step selects every k-th element: `x[..., 0::2]` takes even positions and `x[..., 1::2]` odd ones. RoPE uses exactly this to split vectors into pairs.
`...` (Ellipsis) means "all leading dims", so the same code works for `(S, D)` and `(B, H, S, D)`.

In [ ]:
x = torch.arange(12).view(2, 6)
print(x[:, 1:4])        # columns 1..3
print(x[..., 0::2])     # even columns
print(x[..., 1::2])     # odd columns
print(x[-1])            # last row

## 2. Integer (fancy) indexing

Indexing with an integer **tensor** selects rows: `weight[indices]` is an embedding lookup, and works for any shape of `indices`.
Indexing with **two** index tensors pairs them up element-wise: `x[rows, cols]` returns `x[rows[0], cols[0]], x[rows[1], cols[1]], …`. This is **not** the same as `x[rows][:, cols]`.

In [ ]:
weight = torch.arange(10.).view(5, 2)          # 5 embeddings of size 2
idx = torch.tensor([[0, 4], [2, 2]])
print(weight[idx].shape)                       # (2, 2, 2): idx shape + embedding dim

x = torch.tensor([[10., 11., 12.],
                  [20., 21., 22.]])
print(x[torch.tensor([0, 1]), torch.tensor([2, 0])])   # tensor([12., 20.])

## 3. Boolean masks

- **Reading** `x[mask]` returns a **1-D** tensor of the selected elements. The shape is lost.
- **Writing** `x[mask] = value` (or `x[mask] += ...`) updates only the selected positions.
- `torch.where(mask, a, b)` picks element-wise and **keeps the shape**.

In [ ]:
x = torch.tensor([[1., -2.], [-3., 4.]])
mask = x < 0
print(x[mask])                        # tensor([-2., -3.]): flattened
print(torch.where(mask, 0.0, x))      # same shape, negatives replaced
y = x.clone(); y[mask] = 0.0; print(y)

## 4. `masked_fill` and building masks

`scores.masked_fill(mask, float("-inf"))` returns a copy with `-inf` wherever `mask` is `True`. The mask **broadcasts**, so one `(S, S)` mask can cover a whole `(B, H, S, S)` score tensor. After softmax, `-inf` becomes probability `0`.
Masks are built from comparisons, or with `torch.triu` / `torch.tril`. `diagonal=k` shifts which diagonal is kept.

> ⚠️ If an entire row is `-inf`, softmax returns `NaN` for that row. Make sure every query can see at least one key (P9 covers this).

In [ ]:
S = 4
future = torch.triu(torch.ones(S, S, dtype=torch.bool), diagonal=1)   # True above the diagonal
print(future)
i = torch.arange(S)
print(torch.equal(future, i[None, :] > i[:, None]))   # same mask, from a comparison (P2)

scores = torch.zeros(2, S, S)
print(torch.softmax(scores.masked_fill(future, float("-inf")), dim=-1)[0])

## 5. `topk` and `sort`

Both return `(values, indices)` along a dim (default: last). `topk(k)` gives the `k` largest, in descending order. `sort` orders everything.
The **indices** tell you where each value came from, and you need them to put values back (§6).

In [ ]:
logits = torch.tensor([0.1, 2.0, -1.0, 3.0, 0.5])
vals, idx = logits.topk(2)
print(vals, idx)                    # tensor([3., 2.]) tensor([3, 1])
s_vals, s_idx = logits.sort(descending=True)
print(s_vals, s_idx)

## 6. `gather` and `scatter`

- `x.gather(dim, idx)` **reads**: `out[i][j] = x[i][idx[i][j]]` (for `dim=1`). `out` has the shape of `idx`.
- `x.scatter(dim, idx, src)` **writes**: `out[i][idx[i][j]] = src[i][j]`. It returns a copy of `x` with those positions overwritten (`scatter_` is the in-place version).

They are inverses: `gather` pulls values *from* positions, `scatter` pushes values *to* positions.

In [ ]:
x = torch.tensor([[10., 11., 12.],
                  [20., 21., 22.]])
idx = torch.tensor([[2, 0],
                    [1, 1]])
print(x.gather(1, idx))                          # [[12., 10.], [21., 21.]]

base = torch.zeros(2, 3)
print(base.scatter(1, torch.tensor([[2], [0]]), torch.tensor([[5.], [7.]])))
# [[0., 0., 5.], [7., 0., 0.]]

## 7. Grouping by id

GRPO (#38) normalizes each reward against the other rewards **from the same prompt**. Group ids can be arbitrary integers, like `[7, 3, 7, 10, 3]`.

- `torch.unique(ids, return_inverse=True)` returns `(unique_ids, inverse)`, where `inverse[i]` is the position of `ids[i]` in `unique_ids`. That relabels the groups as `0 … G-1`.
- `torch.bincount(inverse)` counts the members of each group.
- `torch.zeros(G).index_add(0, inverse, values)` adds every value into its group's slot (`bincount(inverse, weights=values)` also works).
- `group_stat[inverse]` broadcasts a per-group result back to every element.

No loop over groups needed, no matter how many there are.

In [ ]:
ids = torch.tensor([7, 3, 7, 10, 3])
vals = torch.tensor([1., 2., 3., 4., 5.])
uniq, inv = ids.unique(return_inverse=True)
print(uniq, inv)                                  # tensor([ 3,  7, 10]) tensor([1, 0, 1, 2, 0])
counts = torch.bincount(inv)
sums = torch.zeros(len(uniq)).index_add(0, inv, vals)
print(counts, sums)                               # [2, 2, 1]  [7., 4., 4.]
print((sums / counts)[inv])                       # each element's group mean: [2., 3.5, 2., 4., 3.5]

## 8. More selection tools

- **`F.one_hot(idx, num_classes)`** turns class ids into one-hot rows (as `int64`; convert with `.float()`).
- **`torch.nonzero(mask)`** / **`torch.where(mask)`** (one argument) return the *positions* of `True` entries. The output size depends on the data, so on a GPU the CPU must wait for the result (a **sync**), and tools like `torch.compile` can't plan around it. Prefer a mask or `masked_fill` when you can.
- **`torch.searchsorted(sorted_seq, values)`** finds where each value would be inserted. With a cumulative distribution it samples by **inverse CDF**: draw `u ~ U(0, 1)` and find the first bucket whose cumulative probability exceeds `u`.
- **`torch.take_along_dim(x, idx, dim)`** is `gather` that also broadcasts `idx`; **`x.index_select(dim, idx)`** picks whole slices along one dim.
- **`scatter_add` / `scatter_reduce`** write **many** values into the **same** slot, combining them (sum, max, mean…). Plain `scatter` would keep only one of them.

In [ ]:
import torch.nn.functional as F
print(F.one_hot(torch.tensor([0, 2, 1]), num_classes=3))
print(torch.nonzero(torch.tensor([[0, 1], [1, 1]])))         # (row, col) of each nonzero

probs = torch.tensor([0.1, 0.6, 0.3])
cdf = probs.cumsum(0)                                         # [0.1, 0.7, 1.0]
u = torch.tensor([0.05, 0.5, 0.95])
print(torch.searchsorted(cdf, u))                             # [0, 1, 2]: inverse-CDF sampling

x = torch.tensor([[10., 11., 12.], [20., 21., 22.]])
print(torch.take_along_dim(x, torch.tensor([[2], [0]]), dim=1))   # [[12.], [20.]]
print(x.index_select(1, torch.tensor([0, 2])))                     # columns 0 and 2

slots = torch.zeros(3).scatter_add(0, torch.tensor([0, 0, 2]), torch.tensor([1., 2., 5.]))
print(slots)                                                  # [3., 0., 5.]: both writes to slot 0 are summed

## 9. Many ways to do it

| Goal | Equivalent forms | Watch out for |
|------|------------------|---------------|
| One value per row | `x[torch.arange(B), idx]`, `x.gather(1, idx[:, None]).squeeze(1)`, `torch.take_along_dim(x, idx[:, None], 1).squeeze(1)`, `(F.one_hot(idx, C) * x).sum(1)` | the one-hot form builds a full `(B, C)` tensor: wasteful for large vocabularies |
| Sums per group | `zeros(G).index_add(0, inv, v)`, `zeros(G).scatter_add(0, inv, v)`, `torch.bincount(inv, weights=v)` | `bincount` has **no gradient**; on GPU, `index_add`/`scatter_add` use atomic adds, so float results can differ slightly run to run |
| Replace where a condition holds | `x.masked_fill(mask, v)`, `torch.where(mask, v, x)`, `x.clone(); y[mask] = v` | the last one is in place on a copy: fine, but easy to forget the `clone()` |
| Sample from probabilities | `torch.multinomial(p, n)`, `torch.searchsorted(p.cumsum(0), torch.rand(n))` | `multinomial` handles rounding at the top end of the CDF; with `searchsorted`, clamp to `len(p) - 1` |

In [ ]:
torch.manual_seed(0)
B, C = 4, 6
x = torch.randn(B, C)
idx = torch.tensor([0, 5, 2, 2])
a = x[torch.arange(B), idx]
b = x.gather(1, idx[:, None]).squeeze(1)
c = (F.one_hot(idx, C) * x).sum(1)
print(torch.equal(a, b), torch.allclose(a, c))

inv = torch.tensor([0, 1, 0, 2])
v = torch.tensor([1., 2., 3., 4.])
print(torch.zeros(3).index_add(0, inv, v), torch.zeros(3).scatter_add(0, inv, v), torch.bincount(inv, weights=v))

---
# ✏️ Exercises

Implement the six functions below. Each one has a debug cell you can use before you submit.

### Exercise 1: `pick_per_row(logits, targets)`

For each row, pick the value at that row's target column.

- `logits`: `(B, C)`
- `targets`: `(B,)` integer class ids
- returns: `(B,)` with `out[b] = logits[b, targets[b]]`

*Cross-entropy (#16) is `-pick_per_row(log_probs, targets).mean()`.*

In [ ]:
# ✅ SOLUTION

def pick_per_row(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    rows = torch.arange(logits.shape[0], device=logits.device)
    return logits[rows, targets]

In [ ]:
logits = torch.tensor([[0.1, 0.7, 0.2],
                       [0.5, 0.3, 0.2]])
print(pick_per_row(logits, torch.tensor([1, 0])))   # expect tensor([0.7000, 0.5000])

### Exercise 2: `attention_mask(n, kind, w=0)`

Build an `(n, n)` **boolean** mask for attention, where `True` means **blocked** (query `i` may *not* attend to key `j`).

- `kind="causal"`: block the future, `j > i`
- `kind="sliding"`: two-sided window (Longformer-style, #11), block `abs(i - j) > w`
- `kind="causal_sliding"`: causal window (Mistral-style), block `j > i` **or** `i - j > w`
- any other `kind`: raise `ValueError`

*Hint: P2's `relative_position(n)` (`R[i, j] = i - j`) makes each kind a single comparison.*

In [ ]:
# ✅ SOLUTION

def attention_mask(n: int, kind: str, w: int = 0) -> torch.Tensor:
    idx = torch.arange(n)
    R = idx[:, None] - idx[None, :]          # R[i, j] = i - j
    if kind == "causal":
        return R < 0
    if kind == "sliding":
        return R.abs() > w
    if kind == "causal_sliding":
        return (R < 0) | (R > w)
    raise ValueError(f"unknown mask kind: {kind!r}")

In [ ]:
for kind in ("causal", "sliding", "causal_sliding"):
    print(kind)
    print(attention_mask(5, kind, w=1).int())
# causal_sliding with w=1 should leave exactly 2 visible keys per row (itself and the previous one),
# except row 0, which only sees itself.

### Exercise 3: `apply_mask(scores, mask)`

Set blocked attention scores to `-inf` so softmax gives them zero probability.

- `scores`: `(..., S_q, S_k)` float, any number of leading dims (batch, heads)
- `mask`: `(S_q, S_k)` bool, `True` = blocked
- returns: same shape and dtype as `scores`
- do **not** modify `scores` in place

In [ ]:
# ✅ SOLUTION

def apply_mask(scores: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return scores.masked_fill(mask, float("-inf"))

In [ ]:
scores = torch.zeros(2, 3, 3)
masked = apply_mask(scores, attention_mask(3, "causal"))
print(masked[0])
print(torch.softmax(masked, dim=-1)[0])   # row i spreads evenly over keys 0..i

### Exercise 4: `keep_topk(logits, k)`

Keep the `k` largest logits in each row (last dim) and set every other entry to `-inf`.

- `logits`: `(..., V)` float
- `k`: int, `1 <= k`; if `k >= V`, keep everything
- returns: same shape; kept entries **unchanged**, others `-inf`
- do **not** modify `logits` in place

*#32 samples only from the top-k tokens; #28 MoE routes each token to its top-k experts.*

In [ ]:
# ✅ SOLUTION

def keep_topk(logits: torch.Tensor, k: int) -> torch.Tensor:
    k = min(k, logits.shape[-1])
    vals, idx = logits.topk(k, dim=-1)
    return torch.full_like(logits, float("-inf")).scatter(-1, idx, vals)

In [ ]:
logits = torch.tensor([[0.1, 2.0, -1.0, 3.0, 0.5],
                       [9.0, 8.0, 7.0, 6.0, 5.0]])
print(keep_topk(logits, 2))
# expect [[-inf, 2., -inf, 3., -inf],
#         [ 9.,  8., -inf, -inf, -inf]]

### Exercise 5: `unsort(sorted_values, sort_idx)`

Undo a sort: given `sorted_values, sort_idx = x.sort(dim=-1)` (possibly after editing `sorted_values`), put each value back at its original position.

- `sorted_values`: `(..., V)`
- `sort_idx`: `(..., V)` long, from `x.sort(dim=-1)`; `sorted_values[..., j]` came from position `sort_idx[..., j]`
- returns: `(..., V)` with `out[..., sort_idx[..., j]] = sorted_values[..., j]`

*Top-p sampling (#32) sorts the logits, masks the tail in sorted order, then has to put them back.*

In [ ]:
# ✅ SOLUTION

def unsort(sorted_values: torch.Tensor, sort_idx: torch.Tensor) -> torch.Tensor:
    return torch.empty_like(sorted_values).scatter(-1, sort_idx, sorted_values)

In [ ]:
x = torch.tensor([0.3, -1.0, 2.0, 0.0])
s, i = x.sort(descending=True)
print(s, i)
print(unsort(s, i))                      # expect the original x
s[2:] = float("-inf")                    # drop the two smallest, in sorted order
print(unsort(s, i))                      # expect tensor([0.3, -inf, 2.0, -inf])

### Exercise 6: `group_mean(values, group_ids)`

Replace every value with the mean of its group.

- `values`: `(B,)` float
- `group_ids`: `(B,)` integer ids; arbitrary values, not necessarily `0 … G-1` or sorted
- returns: `(B,)` with `out[i] = mean of values[j] over all j with group_ids[j] == group_ids[i]`
- same dtype as `values`, gradients must flow back to `values`, and **no Python loop over groups**

*#38 GRPO computes this (and a per-group std) to turn rewards into advantages.*

In [ ]:
# ✅ SOLUTION

def group_mean(values: torch.Tensor, group_ids: torch.Tensor) -> torch.Tensor:
    uniq, inv = torch.unique(group_ids, return_inverse=True)
    sums = torch.zeros(len(uniq), dtype=values.dtype, device=values.device).index_add(0, inv, values)
    counts = torch.bincount(inv, minlength=len(uniq)).to(values.dtype)
    return (sums / counts)[inv]

In [ ]:
print(group_mean(torch.tensor([1., 2., 3., 4., 5.]), torch.tensor([7, 3, 7, 10, 3])))
# expect tensor([2.0000, 3.5000, 2.0000, 4.0000, 3.5000])

<!-- Generated from scripts/prereqs/content/p04_indexing_masking.py by `make prereqs`. Edit that file instead of this notebook; manual edits are overwritten. -->
## ✅ Submit

Run the judge on **all** exercises in this notebook. Each test is labelled with the exercise it checks.

In [ ]:
from torch_judge import check, hint
check("p04_indexing_masking")

---
## 🔓 What this unlocks

With **Indexing & Masking** in your toolkit, you're ready for these problems:

| # | Problem | Difficulty | What you'll reuse | Links |
|:---:|---|:---:|---|---|
| #9 | Causal Self-Attention | 🔴 Hard | `attention_mask(n, 'causal')` plus `apply_mask` | [Open](09_causal_attention.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/09_causal_attention.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/09_causal_attention.ipynb" target="_blank">Colab</a> |
| #11 | Sliding Window Attention | 🔴 Hard | `attention_mask(n, 'sliding', w)`; `'causal_sliding'` is the Mistral variant | [Open](11_sliding_window.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/11_sliding_window.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/11_sliding_window.ipynb" target="_blank">Colab</a> |
| #13 | GPT-2 Block | 🔴 Hard | The causal mask and `masked_fill`, inside a full transformer block | [Open](13_gpt2_block.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/13_gpt2_block.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/13_gpt2_block.ipynb" target="_blank">Colab</a> |
| #14 | KV Cache Attention | 🔴 Hard | A causal mask shifted by the cache length: `torch.triu(..., diagonal=S_past + 1)` (lesson §4) | [Open](14_kv_cache.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/14_kv_cache.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/14_kv_cache.ipynb" target="_blank">Colab</a> |
| #16 | Cross-Entropy Loss | 🟢 Easy | `pick_per_row`: the log-probability of the correct class | [Open](16_cross_entropy.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/16_cross_entropy.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/16_cross_entropy.ipynb" target="_blank">Colab</a> |
| #18 | Embedding | 🟢 Easy | `weight[indices]`: the embedding lookup (lesson §2) | [Open](18_embedding.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/18_embedding.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/18_embedding.ipynb" target="_blank">Colab</a> |
| #24 | RoPE | 🔴 Hard | Strided slicing `x[..., 0::2]` and `x[..., 1::2]` for RoPE pairs (lesson §1) | [Open](24_rope.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/24_rope.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/24_rope.ipynb" target="_blank">Colab</a> |
| #28 | Mixture of Experts | 🔴 Hard | `topk` routing, then boolean-mask reads and writes like `output[mask] += ...` (lesson §3) | [Open](28_moe.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/28_moe.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/28_moe.ipynb" target="_blank">Colab</a> |
| #32 | Top-k / Top-p Sampling | 🟡 Medium | `keep_topk` for top-k, then `sort`, mask, and `unsort` for top-p | [Open](32_topk_sampling.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/32_topk_sampling.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/32_topk_sampling.ipynb" target="_blank">Colab</a> |
| #38 | GRPO Loss | 🔴 Hard | `group_mean`: per-group statistics with `unique` / `bincount` / `index_add`, no loop over groups | [Open](38_grpo_loss.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/38_grpo_loss.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/38_grpo_loss.ipynb" target="_blank">Colab</a> |

🗺️ See the full recommended order in <a href="https://github.com/lalo/TorchCode/blob/master/LEARNING_PATH.md" target="_blank">LEARNING_PATH.md</a>.